# T Cell CD4/CD8 Classification + scANVI Re-training (v1.3 PRODUCTION)

## Critical Fixes in v1.3

**Must-Fix Issues (from code review):**
1. ✅ **CD4/CD8 classification**: Winner-takes-all (no DP/DN prefixes)
2. ✅ **log1p layer**: Created in Stage 0 with layer in-place API
3. ✅ **HVG consistency**: Load fixed HVG list from pre-trained model
4. ✅ **Regex escaping**: Safe keyword matching
5. ✅ **Gene matching**: Handle duplicate symbols robustly

**Pipeline Logic:**
- Filter out: Monocyte, Macrophage, B cells
- Keep: T cells + NK cells
- CD4/CD8 labels: **T cells only** (winner-takes-all)
- NK cells: Keep original CellTypist label

---

Author: r2end  
Date: 2025-01-09  
Version: 1.3 (Production hotfix)

## Imports

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
import scanpy as sc
import matplotlib
matplotlib.use('Agg')  # Headless server compatible
import matplotlib.pyplot as plt
import seaborn as sns
import time
import gc
import re
from scipy.sparse import issparse

import scvi
import torch

print("Libraries imported successfully")

## Configuration

In [ ]:
# ============================================================================
# INPUT/OUTPUT PATHS
# ============================================================================

INPUT_FILE = "/home/h2048/data/py/0108/tcell_scanvi_v3_5_1_hotfix/adata_epithelial_FINAL.h5ad"
SCVI_MODEL_PATH = "/home/h2048/data/py/0108/tcell_scanvi_v3_5_1_hotfix/models/scvi_model"
OUTPUT_DIR = "/home/h2048/data/py/0109/tcell_cd4cd8_scanvi_v1_3"

# ============================================================================
# CELL TYPE FILTERING
# ============================================================================

# Remove these keywords (will use regex escaping)
EXCLUDE_KEYWORDS = [
    # Myeloid cells
    'Monocyte', 'Macrophage', 'myeloid','Mono-mac',
    # B cells  
    'B cell', 'B-cell', 'plasma','Naive B cells'
]

# T cell identification (for CD4/CD8 labeling)
T_CELL_KEYWORDS = [
    'T cell', 'T-cell', 'CD4', 'CD8', 'Treg', 'Tem', 'Tcm', 'Teff', 'Tn',
    'naive T', 'memory T', 'effector T', 'regulatory T'
]

# ============================================================================
# CD4/CD8 CLASSIFICATION (WINNER-TAKES-ALL)
# ============================================================================

CD4_GENE = 'CD4'
CD8A_GENE = 'CD8A'
CD8B_GENE = 'CD8B'
CD4CD8_THRESHOLD = 1.0  # log1p normalized expression

# CD8 score = max(CD8A, CD8B)
# Classification: winner-takes-all (CD4 vs CD8)
# - If both < threshold OR both > threshold (DP) → no prefix (avoid label explosion)
# - Only clear winner gets prefix: CD4+ or CD8+

# ============================================================================
# SCANVI TRAINING
# ============================================================================

SCANVI_MAX_EPOCHS = 600
BATCH_SIZE = 2048
LEARNING_RATE = 1e-3
EARLY_STOPPING = True
EARLY_STOPPING_PATIENCE = 50

# ============================================================================
# RARE TYPE FILTERING
# ============================================================================

MIN_CELLS_PER_TYPE = 50

# ============================================================================
# COLUMN KEYS
# ============================================================================

BATCH_KEY = 'dataset'
CELLTYPIST_KEY = 'cell_type_celltypist_raw'
CD4CD8_KEY = 'cd4_cd8_type'
COMBINED_LABEL_KEY = 'cell_type_combined'
SCANVI_LABEL_KEY = 'cell_type_scanvi_cd4cd8'

RANDOM_SEED = 42

print("="*80)
print("T CELL CD4/CD8 CLASSIFICATION v1.3 PRODUCTION")
print("="*80)
print(f"\nConfiguration:")
print(f"  Input: {INPUT_FILE}")
print(f"  Output: {OUTPUT_DIR}")
print(f"  scVI model: {SCVI_MODEL_PATH}")
print(f"  CD4/CD8 strategy: Winner-takes-all")
print(f"  Rare type threshold: {MIN_CELLS_PER_TYPE} cells")

## Helper Functions

In [ ]:
def merge_rare_types(s: pd.Series, min_cells: int = 50, other: str = "Unknown") -> pd.Series:
    """Merge rare cell types (< min_cells) into 'other' category."""
    s = s.astype(str).copy()
    vc = s.value_counts()
    rare = vc[vc < min_cells].index
    
    if len(rare) > 0:
        print(f"   Merging {len(rare)} rare types (<{min_cells} cells) → {other}")
        for rt in rare[:5]:
            print(f"      {rt}: {vc[rt]} cells")
        if len(rare) > 5:
            print(f"      ... and {len(rare)-5} more")
        s.loc[s.isin(rare)] = other
    
    return s


def save_celltype_counts(counts_dict: dict, output_dir: Path) -> None:
    """Save cell type count statistics to CSV files"""
    for name, series in counts_dict.items():
        df = pd.DataFrame({
            'cell_type': series.index,
            'count': series.values,
            'percentage': (series.values / series.sum() * 100).round(2)
        })
        df = df.sort_values('count', ascending=False)
        filepath = output_dir / f"{name}.csv"
        df.to_csv(filepath, index=False)
        print(f"   ✓ Saved: {filepath.name}")


def is_tcell(cell_type: str, keywords: list) -> bool:
    """Check if a cell type matches T cell keywords"""
    cell_type_lower = str(cell_type).lower()
    return any(kw.lower() in cell_type_lower for kw in keywords)


print("Helper functions defined")

## Random Seeds

In [ ]:
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

scvi.settings.seed = RANDOM_SEED
print(f"✓ Random seed set to {RANDOM_SEED}")

## Environment Setup

In [ ]:
# GPU check
print(f"\n{'='*80}")
print("GPU CHECK")
print("="*80)

GPU_AVAILABLE = torch.cuda.is_available()
print(f"GPU available: {GPU_AVAILABLE}")

if GPU_AVAILABLE:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name}")
    print(f"Memory: {gpu_memory:.1f} GB")
    accelerator = 'gpu'
    devices = 1
else:
    print("⚠️  Running on CPU")
    accelerator = 'cpu'
    devices = 'auto'

# Create directories
OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)
MODEL_DIR = OUTPUT_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)

print(f"\n✓ Output directories created")

# Configure scanpy
sc.settings.verbosity = 3
sc.settings.n_jobs = 48
sc.settings.figdir = FIG_DIR
sc.set_figure_params(dpi=300, facecolor='white', format='pdf')

## STAGE 0: Data Loading + log1p Creation

In [ ]:
print(f"\n{'='*80}")
print("STAGE 0: DATA LOADING")
print("="*80)

print(f"\nLoading: {INPUT_FILE}")
adata = sc.read_h5ad(INPUT_FILE)

print(f"✓ Data loaded:")
print(f"  Cells: {adata.n_obs:,}")
print(f"  Genes: {adata.n_vars:,}")

# Critical assertions
assert BATCH_KEY in adata.obs.columns, f"Missing batch key: {BATCH_KEY}"
assert CELLTYPIST_KEY in adata.obs.columns, f"Missing CellTypist key: {CELLTYPIST_KEY}"
assert 'X_scvi' in adata.obsm, "Missing X_scvi - need pre-trained scVI"
assert 'counts' in adata.layers, "Missing counts layer"
assert adata.obs_names.is_unique, "Non-unique cell barcodes"

print(f"\n✓ Required structures validated:")
print(f"  Batch key: {BATCH_KEY}")
print(f"  CellTypist labels: {CELLTYPIST_KEY}")
print(f"  X_scvi: {adata.obsm['X_scvi'].shape}")
print(f"  Layers: {list(adata.layers.keys())}")
print(f"  .raw: {adata.raw is not None}")

In [ ]:
# ⭐ FIX #2: Create log1p layer NOW (before CD4/CD8 classification)
# Use layer in-place API for safety
print(f"\n{'='*80}")
print("CREATE log1p LAYER (for CD4/CD8 threshold)")
print("="*80)

if 'log1p' in adata.layers:
    print(f"  ✓ log1p layer already exists")
else:
    print(f"\n  Creating log1p from counts...")
    adata.layers['log1p'] = adata.layers['counts'].copy()
    
    # Use scanpy's layer-aware normalization
    sc.pp.normalize_total(adata, target_sum=1e4, layer='log1p')
    sc.pp.log1p(adata, layer='log1p')
    
    print(f"  ✓ log1p layer created")

# Set X to log1p for downstream use
adata.X = adata.layers['log1p']
print(f"  ✓ adata.X set to log1p")

## STAGE 1: Filter Non-T/NK Cells

In [ ]:
print(f"\n{'='*80}")
print("STAGE 1: FILTER NON-T/NK CELLS")
print("="*80)

print(f"\nCell type distribution (before filtering):")
celltypist_counts_before = adata.obs[CELLTYPIST_KEY].value_counts()
print(celltypist_counts_before.head(20))

# ⭐ FIX #4: Use regex escaping for safe pattern matching
print(f"\nIdentifying cells to exclude (safe regex)...")
exclude_pattern = '|'.join(re.escape(k) for k in EXCLUDE_KEYWORDS)
exclude_mask = adata.obs[CELLTYPIST_KEY].astype(str).str.contains(
    exclude_pattern,
    case=False,
    regex=True,
    na=False
)

n_exclude = exclude_mask.sum()
pct_exclude = n_exclude / adata.n_obs * 100

print(f"\nCells to exclude: {n_exclude:,} ({pct_exclude:.1f}%)")

if n_exclude > 0:
    print(f"\nCell types to remove:")
    exclude_types = adata.obs.loc[exclude_mask, CELLTYPIST_KEY].value_counts()
    for ct, count in exclude_types.items():
        print(f"  {ct}: {count:,} cells")

# Filter
print(f"\nFiltering out excluded cells...")
adata = adata[~exclude_mask].copy()

print(f"\n✓ Data after filtering:")
print(f"  Cells: {adata.n_obs:,}")
print(f"  Removed: {n_exclude:,} cells")

print(f"\nRemaining cell types:")
celltypist_counts_after = adata.obs[CELLTYPIST_KEY].value_counts()
print(celltypist_counts_after.head(20))

## STAGE 2: CD4/CD8 Classification (Winner-Takes-All)

In [ ]:
from scipy.sparse import issparse

def _as_int_list(idxs):
    """idxs can be int / list / np.ndarray; return python list[int]."""
    if isinstance(idxs, (np.ndarray, list, tuple)):
        return [int(i) for i in list(idxs)]
    return [int(idxs)]

def extract_expr_1d_from_layer(adata, layer_key, col_idxs, agg="max"):
    """
    Extract expression vector from adata.layers[layer_key] for given columns.
    Always returns 1D float ndarray (shape = n_cells,).
    
    - If multiple columns: aggregate across columns (default=max).
    - Works for sparse and dense matrices.
    """
    cols = _as_int_list(col_idxs)
    X = adata.layers[layer_key][:, cols]  # (n_cells, k)

    if issparse(X):
        if X.shape[1] == 1:
            # sparse (n,1) -> dense 1D
            return np.asarray(X.toarray()).ravel().astype(np.float32)
        else:
            if agg == "max":
                return np.asarray(X.max(axis=1)).ravel().astype(np.float32)
            elif agg == "mean":
                return np.asarray(X.mean(axis=1)).ravel().astype(np.float32)
            else:
                raise ValueError(f"Unsupported agg={agg}")
    else:
        X = np.asarray(X)
        if X.ndim == 1:
            return X.ravel().astype(np.float32)
        if X.shape[1] == 1:
            return X[:, 0].ravel().astype(np.float32)
        if agg == "max":
            return X.max(axis=1).ravel().astype(np.float32)
        elif agg == "mean":
            return X.mean(axis=1).ravel().astype(np.float32)
        else:
            raise ValueError(f"Unsupported agg={agg}")


In [ ]:
# ============================================================================
# STAGE 2: CD4/CD8 CLASSIFICATION (USE FULL ADATA, NOT HVG SUBSET)
# ============================================================================
# REPLACE the original STAGE 2 with this version

print(f"\n{'='*80}")
print("STAGE 2: CD4/CD8 CLASSIFICATION (FULL GENE SET)")
print("="*80)

# Get gene symbol column (from FULL adata, not HVG subset)
print(f"\nLocating gene symbols in FULL adata...")

gene_symbols = None
symbol_col = None

for col in ['symbol_base', 'gene_symbol', 'gene_symbols', 'feature_name']:
    if col in adata.var.columns:
        gene_symbols = adata.var[col].astype(str)
        symbol_col = col
        break

if gene_symbols is None:
    gene_symbols = adata.var_names.astype(str)
    symbol_col = 'var_names'

print(f"  Gene symbols from: {symbol_col}")
print(f"  Total genes available: {len(gene_symbols)}")

# Define function to find genes (case-insensitive)
def find_gene_flexible(gene_name, gene_symbols):
    """Find gene with case-insensitive matching"""
    # Try exact match first
    matches = np.where(gene_symbols == gene_name)[0]
    if len(matches) > 0:
        return matches
    
    # Try case-insensitive
    matches = np.where(gene_symbols.str.upper() == gene_name.upper())[0]
    return matches

# Find CD4, CD8A, CD8B
print(f"\nSearching for marker genes:")

cd4_matches = find_gene_flexible(CD4_GENE, gene_symbols)
cd8a_matches = find_gene_flexible(CD8A_GENE, gene_symbols)
cd8b_matches = find_gene_flexible(CD8B_GENE, gene_symbols)

# Check results
if len(cd4_matches) == 0:
    print(f"  ✗ {CD4_GENE}: NOT FOUND")
    raise ValueError(f"Gene {CD4_GENE} not found in adata.var")
else:
    print(f"  ✓ {CD4_GENE}: {len(cd4_matches)} match(es) at index {cd4_matches.tolist()}")

if len(cd8a_matches) == 0:
    print(f"  ✗ {CD8A_GENE}: NOT FOUND")
    raise ValueError(f"Gene {CD8A_GENE} not found in adata.var")
else:
    print(f"  ✓ {CD8A_GENE}: {len(cd8a_matches)} match(es) at index {cd8a_matches.tolist()}")

has_cd8b = len(cd8b_matches) > 0
if has_cd8b:
    print(f"  ✓ {CD8B_GENE}: {len(cd8b_matches)} match(es) at index {cd8b_matches.tolist()}")
else:
    print(f"  ⚠️  {CD8B_GENE}: not found (will use CD8A only)")

# Extract expression from log1p layer (full gene set)
print(f"\nExtracting marker expression from log1p layer...")
print(f"  log1p layer shape: {adata.layers['log1p'].shape}")

# CD4
cd4_expr = extract_expr_1d_from_layer(adata, 'log1p', cd4_matches, agg="max")

# CD8A
cd8a_expr = extract_expr_1d_from_layer(adata, 'log1p', cd8a_matches, agg="max")

# CD8B (optional)
if has_cd8b:
    cd8b_expr = extract_expr_1d_from_layer(adata, 'log1p', cd8b_matches, agg="max")
    cd8_expr = np.maximum(cd8a_expr, cd8b_expr).astype(np.float32)
    print(f"  CD8 score: max(CD8A, CD8B)")
else:
    cd8_expr = cd8a_expr.astype(np.float32)
    print(f"  CD8 score: CD8A only")


# Store in obs
adata.obs['CD4_expr'] = cd4_expr
adata.obs['CD8A_expr'] = cd8a_expr
if has_cd8b:
    adata.obs['CD8B_expr'] = cd8b_expr
adata.obs['CD8_score'] = cd8_expr

print(f"\nExpression statistics:")
print(f"  CD4:  mean={cd4_expr.mean():.2f}, median={np.median(cd4_expr):.2f}, max={cd4_expr.max():.2f}")
print(f"  CD8A: mean={cd8a_expr.mean():.2f}, median={np.median(cd8a_expr):.2f}, max={cd8a_expr.max():.2f}")
if has_cd8b:
    print(f"  CD8B: mean={cd8b_expr.mean():.2f}, median={np.median(cd8b_expr):.2f}, max={cd8b_expr.max():.2f}")
print(f"  CD8 score: mean={cd8_expr.mean():.2f}, median={np.median(cd8_expr):.2f}, max={cd8_expr.max():.2f}")

# Identify T cells
print(f"\nIdentifying T cells (for CD4/CD8 labeling)...")

is_t_cell = adata.obs[CELLTYPIST_KEY].apply(
    lambda x: is_tcell(x, T_CELL_KEYWORDS)
)

n_tcells = is_t_cell.sum()
n_non_tcells = (~is_t_cell).sum()

print(f"  T cells: {n_tcells:,} ({n_tcells/adata.n_obs*100:.1f}%)")
print(f"  Non-T cells (e.g., NK): {n_non_tcells:,} ({n_non_tcells/adata.n_obs*100:.1f}%)")

# Winner-takes-all classification
print(f"\nClassifying T cells (winner-takes-all, threshold: {CD4CD8_THRESHOLD})...")

mask_t = is_t_cell.values
delta = cd4_expr - cd8_expr

# Initialize all as empty string (no prefix)
cd4cd8_type = np.array([''] * adata.n_obs, dtype=object)

# CD4+ : T cell AND CD4 > threshold AND CD4 > CD8
mask_cd4 = mask_t & (cd4_expr > CD4CD8_THRESHOLD) & (delta > 0)

# CD8+ : T cell AND CD8 > threshold AND CD8 > CD4
mask_cd8 = mask_t & (cd8_expr > CD4CD8_THRESHOLD) & (delta < 0)

cd4cd8_type[mask_cd4] = 'CD4+'
cd4cd8_type[mask_cd8] = 'CD8+'

# For non-T cells, keep empty (no CD4/CD8 prefix)
adata.obs[CD4CD8_KEY] = cd4cd8_type
adata.obs['is_t_cell'] = is_t_cell

# Statistics (T cells only)
tcell_cd4cd8 = adata.obs.loc[is_t_cell, CD4CD8_KEY].value_counts()

print(f"\n✓ T cell CD4/CD8 classification (winner-takes-all):")
for cat in ['CD4+', 'CD8+', '']:
    if cat in tcell_cd4cd8.index:
        count = tcell_cd4cd8[cat]
        pct = count / n_tcells * 100
        label = cat if cat else 'Unclear (DP/DN/low)'
        print(f"  {label}: {count:,} cells ({pct:.1f}% of T cells)")

print(f"\n  Non-T cells (no CD4/CD8 label): {n_non_tcells:,}")

# Distribution check
print(f"\nExpression distribution in classified cells:")
if (cd4cd8_type == 'CD4+').sum() > 0:
    cd4_in_cd4pos = cd4_expr[cd4cd8_type == 'CD4+']
    print(f"  CD4+ cells - CD4 expr: mean={cd4_in_cd4pos.mean():.2f}, median={np.median(cd4_in_cd4pos):.2f}")

if (cd4cd8_type == 'CD8+').sum() > 0:
    cd8_in_cd8pos = cd8_expr[cd4cd8_type == 'CD8+']
    print(f"  CD8+ cells - CD8 expr: mean={cd8_in_cd8pos.mean():.2f}, median={np.median(cd8_in_cd8pos):.2f}")

print(f"\n✓ Stage 2 complete")

In [ ]:
# Identify T cells
print(f"\nIdentifying T cells (for CD4/CD8 labeling)...")

is_t_cell = adata.obs[CELLTYPIST_KEY].apply(
    lambda x: is_tcell(x, T_CELL_KEYWORDS)
)

n_tcells = is_t_cell.sum()
n_non_tcells = (~is_t_cell).sum()

print(f"  T cells: {n_tcells:,} ({n_tcells/adata.n_obs*100:.1f}%)")
print(f"  Non-T cells (e.g., NK): {n_non_tcells:,} ({n_non_tcells/adata.n_obs*100:.1f}%)")

# ⭐ FIX #2: Winner-takes-all classification
# Only clear winner (CD4 vs CD8) gets prefix
# DP (both high) or DN (both low) → no prefix (avoid label explosion)
print(f"\nClassifying T cells (winner-takes-all, threshold: {CD4CD8_THRESHOLD})...")

mask_t = is_t_cell.values
delta = cd4_expr - cd8_expr

# Initialize all as empty string (no prefix)
cd4cd8_type = np.array([''] * adata.n_obs, dtype=object)

# CD4+ : T cell AND CD4 > threshold AND CD4 > CD8
mask_cd4 = mask_t & (cd4_expr > CD4CD8_THRESHOLD) & (delta > 0)

# CD8+ : T cell AND CD8 > threshold AND CD8 > CD4
mask_cd8 = mask_t & (cd8_expr > CD4CD8_THRESHOLD) & (delta < 0)

cd4cd8_type[mask_cd4] = 'CD4+'
cd4cd8_type[mask_cd8] = 'CD8+'

# For non-T cells, keep empty (no CD4/CD8 prefix)
adata.obs[CD4CD8_KEY] = cd4cd8_type
adata.obs['is_t_cell'] = is_t_cell

# Statistics (T cells only)
tcell_cd4cd8 = adata.obs.loc[is_t_cell, CD4CD8_KEY].value_counts()

print(f"\n✓ T cell CD4/CD8 classification (winner-takes-all):")
for cat in ['CD4+', 'CD8+', '']:
    if cat in tcell_cd4cd8.index:
        count = tcell_cd4cd8[cat]
        pct = count / n_tcells * 100
        label = cat if cat else 'Unclear (DP/DN/low)'
        print(f"  {label}: {count:,} cells ({pct:.1f}% of T cells)")

print(f"\n  Non-T cells (no CD4/CD8 label): {n_non_tcells:,}")

## STAGE 3: Create Combined Labels

In [ ]:
print(f"\n{'='*80}")
print("STAGE 3: CREATE COMBINED LABELS")
print("="*80)

print(f"\nCreating combined labels...")
print(f"  T cells with CD4/CD8: '<CD4/CD8> <CellTypist>'")
print(f"  T cells without CD4/CD8: '<CellTypist>' (DP/DN/low expression)")
print(f"  Non-T cells: '<CellTypist>' (no prefix)")

celltypist_labels = adata.obs[CELLTYPIST_KEY].astype(str)
cd4cd8_prefix = adata.obs[CD4CD8_KEY].astype(str)

# Create combined labels
combined = []
for prefix, label in zip(cd4cd8_prefix, celltypist_labels):
    if prefix and prefix != '':
        # Has CD4/CD8 prefix (only clear winner)
        combined.append(f"{prefix} {label}")
    else:
        # No prefix (NK, DP, DN, low expression)
        combined.append(label)

adata.obs[COMBINED_LABEL_KEY] = pd.Series(combined, index=adata.obs_names)

# Show examples
combined_counts = adata.obs[COMBINED_LABEL_KEY].value_counts()
print(f"\n✓ Combined labels created: {len(combined_counts)} unique types")
print(f"\nTop 20 combined cell types:")
for ct, count in combined_counts.head(20).items():
    print(f"  {ct}: {count:,} cells")

# Save raw counts
print(f"\nSaving raw combined label counts...")
save_celltype_counts(
    {'combined_labels_raw': combined_counts},
    OUTPUT_DIR
)

In [ ]:
# Filter rare types
print(f"\nFiltering rare combined types (min {MIN_CELLS_PER_TYPE} cells)...")
combined_filt = merge_rare_types(
    adata.obs[COMBINED_LABEL_KEY],
    min_cells=MIN_CELLS_PER_TYPE,
    other='Unknown'
)

adata.obs['cell_type_combined_filt'] = combined_filt

# Convert to categorical for scANVI
labels = combined_filt.astype('category')
if 'Unknown' not in labels.cat.categories:
    labels = labels.cat.add_categories(['Unknown'])

adata.obs['labels_for_scanvi'] = labels

# Statistics
n_unknown = int((labels == 'Unknown').sum())
n_labeled = int(adata.n_obs - n_unknown)
n_unique_types = int(labels.nunique()) - (1 if 'Unknown' in labels.cat.categories else 0)

print(f"\n✓ scANVI labels prepared:")
print(f"  Labeled cells: {n_labeled:,} ({n_labeled/adata.n_obs*100:.1f}%)")
print(f"  Unknown cells: {n_unknown:,} ({n_unknown/adata.n_obs*100:.1f}%)")
print(f"  Unique types (excl. Unknown): {n_unique_types}")

# Save filtered counts
print(f"\nSaving filtered combined label counts...")
save_celltype_counts(
    {'combined_labels_filtered': adata.obs['cell_type_combined_filt'].value_counts()},
    OUTPUT_DIR
)

## STAGE 4: Prepare adata_model with Fixed HVG

In [ ]:
print(f"\n{'='*80}")
print("STAGE 4: PREPARE adata_model (WITH HVG CONSISTENCY CHECK)")
print("="*80)

# ⭐ FIX #3: Load fixed HVG list from pre-trained model
print(f"\nLoading HVG list from scVI model...")

hvg_file = Path(SCVI_MODEL_PATH) / "var_names.csv"
alt_hvg_file = Path(SCVI_MODEL_PATH) / "hvg_genes.txt"

hvg_genes = None

if hvg_file.exists():
    print(f"  Found: {hvg_file.name}")
    hvg_genes = pd.read_csv(hvg_file, header=None)[0].tolist()
elif alt_hvg_file.exists():
    print(f"  Found: {alt_hvg_file.name}")
    hvg_genes = pd.read_csv(alt_hvg_file, header=None)[0].tolist()
else:
    print(f"  ⚠️  No HVG file found, falling back to adata.var['highly_variable']")
    if 'highly_variable' in adata.var.columns:
        hvg_mask = adata.var['highly_variable'].values
        hvg_genes = adata.var_names[hvg_mask].tolist()
    else:
        raise ValueError("No HVG info available")

print(f"  ✓ HVG list loaded: {len(hvg_genes)} genes")

# Subset to HVG (with order preservation)
print(f"\nSubsetting adata to HVG genes...")
hvg_mask = adata.var_names.isin(hvg_genes)
n_overlap = hvg_mask.sum()

print(f"  Overlap: {n_overlap} / {len(hvg_genes)} genes ({n_overlap/len(hvg_genes)*100:.1f}%)")

if n_overlap < len(hvg_genes) * 0.95:
    print(f"  ⚠️  Warning: Low HVG overlap, model may not work properly")

# Build adata_model (minimal construction)
adata_hvg = adata[:, hvg_mask].copy()

# ⭐ Reorder to match hvg_genes order
hvg_genes_present = [g for g in hvg_genes if g in adata_hvg.var_names]
adata_hvg = adata_hvg[:, hvg_genes_present].copy()

X_hvg = adata_hvg.layers['counts'].copy()
obs_hvg = adata.obs[[BATCH_KEY, 'labels_for_scanvi']].copy()
var_hvg = adata_hvg.var[[]].copy()

adata_model = sc.AnnData(X=X_hvg, obs=obs_hvg, var=var_hvg)

# ⭐ FIX #6: Don't duplicate counts layer (share reference)
adata_model.layers['counts'] = adata_model.X

print(f"\n✓ adata_model created:")
print(f"  Shape: {adata_model.n_obs:,} cells × {adata_model.n_vars:,} genes")
print(f"  Gene order: Matches pre-trained model")
print(f"  obs columns: {list(adata_model.obs.columns)}")

del adata_hvg, X_hvg, obs_hvg, var_hvg
gc.collect()

## STAGE 5: Load Pre-trained scVI Model

In [ ]:
print(f"\n{'='*80}")
print("STAGE 5: LOAD PRE-TRAINED scVI MODEL")
print("="*80)

# Setup scVI
print(f"\nSetting up scVI on adata_model...")
scvi.model.SCVI.setup_anndata(
    adata_model,
    layer='counts',
    batch_key=BATCH_KEY
)
print(f"✓ scVI setup complete")

# Load model
print(f"\nLoading pre-trained scVI model: {SCVI_MODEL_PATH}")
if not Path(SCVI_MODEL_PATH).exists():
    raise FileNotFoundError(f"scVI model not found: {SCVI_MODEL_PATH}")

vae = scvi.model.SCVI.load(str(SCVI_MODEL_PATH), adata=adata_model)
print(f"✓ scVI model loaded")

# Verify
print(f"\nVerifying latent representation...")
latent_test = vae.get_latent_representation()
print(f"  Latent shape: {latent_test.shape}")
print(f"  Matches adata cells: {latent_test.shape[0] == adata.n_obs}")

del latent_test

## STAGE 6: Train scANVI with Combined Labels

In [ ]:
print(f"\n{'='*80}")
print("STAGE 6: TRAIN scANVI")
print("="*80)

print(f"\nInitializing scANVI from scVI...")
lvae = scvi.model.SCANVI.from_scvi_model(
    vae,
    adata=adata_model,
    labels_key='labels_for_scanvi',
    unlabeled_category='Unknown'
)
print(f"✓ scANVI initialized")

# Train
print(f"\n{'='*80}")
print(f"Training scANVI...")
print(f"{'='*80}\n")

start_time = time.time()
train_kwargs = {
    'max_epochs': SCANVI_MAX_EPOCHS,
    'batch_size': BATCH_SIZE,
    'train_size': 0.9,
    'accelerator': accelerator,
    'devices': devices,
    'plan_kwargs': {'lr': LEARNING_RATE},
}
if EARLY_STOPPING:
    train_kwargs['early_stopping'] = True
    train_kwargs['early_stopping_patience'] = EARLY_STOPPING_PATIENCE

lvae.train(**train_kwargs)
elapsed = time.time() - start_time
print(f"\n✓ Training complete: {int(elapsed//60)}m {int(elapsed%60)}s")

# Save
scanvi_model_dir = MODEL_DIR / "scanvi_cd4cd8_model"
print(f"\nSaving scANVI model: {scanvi_model_dir}")
lvae.save(scanvi_model_dir, overwrite=True)

## STAGE 7: Extract scANVI Results

In [ ]:
print(f"\n{'='*80}")
print("STAGE 7: EXTRACT scANVI RESULTS")
print("="*80)

print(f"\nGenerating scANVI predictions (index-aligned)...")

# Predictions
pred = pd.Series(lvae.predict(), index=adata_model.obs_names)
adata.obs[SCANVI_LABEL_KEY] = pred.reindex(adata.obs_names).astype(str).values

# Confidence
probs = np.asarray(lvae.predict(soft=True))
conf = pd.Series(probs.max(axis=1), index=adata_model.obs_names)
adata.obs['scanvi_cd4cd8_confidence'] = conf.reindex(adata.obs_names).values

# Latent
z = pd.DataFrame(lvae.get_latent_representation(), index=adata_model.obs_names)
adata.obsm['X_scanvi_cd4cd8'] = z.reindex(adata.obs_names).to_numpy()

print(f"✓ scANVI predictions complete")
print(f"  X_scanvi_cd4cd8: {adata.obsm['X_scanvi_cd4cd8'].shape}")
print(f"  Unique types: {adata.obs[SCANVI_LABEL_KEY].nunique()}")
print(f"  Mean confidence: {adata.obs['scanvi_cd4cd8_confidence'].mean():.3f}")

# Filter rare types
print(f"\nFiltering rare types in scANVI predictions...")
scanvi_filt = merge_rare_types(
    adata.obs[SCANVI_LABEL_KEY],
    min_cells=MIN_CELLS_PER_TYPE,
    other='Unknown'
)
adata.obs['cell_type_scanvi_cd4cd8_filt'] = scanvi_filt.astype('category')

print(f"✓ Unique types (filtered): {adata.obs['cell_type_scanvi_cd4cd8_filt'].nunique()}")

# Save counts
print(f"\nSaving scANVI counts...")
save_celltype_counts(
    {
        'scanvi_cd4cd8_counts_raw': adata.obs[SCANVI_LABEL_KEY].value_counts(),
        'scanvi_cd4cd8_counts_filt': adata.obs['cell_type_scanvi_cd4cd8_filt'].value_counts()
    },
    OUTPUT_DIR
)

# Clean up
del adata_model, vae, lvae
gc.collect()

print(f"\n✓ Stage 7 complete")

## STAGE 8: Visualization

In [ ]:
print(f"\n{'='*80}")
print("STAGE 8: VISUALIZATION")
print("="*80)

# Compute UMAP
print(f"\nComputing UMAP on scANVI latent space...")
sc.pp.neighbors(
    adata,
    use_rep='X_scanvi_cd4cd8',
    n_neighbors=15,
    key_added='neighbors_scanvi_cd4cd8'
)
sc.tl.umap(adata, neighbors_key='neighbors_scanvi_cd4cd8')
adata.obsm['X_umap_scanvi_cd4cd8'] = adata.obsm['X_umap'].copy()
print(f"✓ UMAP computed: {adata.obsm['X_umap_scanvi_cd4cd8'].shape}")

### Plot 1: CD4/CD8 Classification

In [ ]:
print(f"\nGenerating CD4/CD8 classification plot...")
fig, axes = plt.subplots(1, 4, figsize=(24, 6))

# ⭐ FIX #7: Use view instead of copy for plotting
adata_tcell = adata[adata.obs['is_t_cell']]

sc.pl.embedding(
    adata_tcell,
    basis='umap_scanvi_cd4cd8',
    color=CD4CD8_KEY,
    ax=axes[0],
    show=False,
    title='CD4/CD8 (T cells, winner-takes-all)',
    size=3,
    legend_loc='right margin',
    frameon=False
)

sc.pl.embedding(
    adata_tcell,
    basis='umap_scanvi_cd4cd8',
    color='CD4_expr',
    ax=axes[1],
    show=False,
    title='CD4 Expression',
    size=2,
    cmap='Reds',
    frameon=False
)

sc.pl.embedding(
    adata_tcell,
    basis='umap_scanvi_cd4cd8',
    color='CD8A_expr',
    ax=axes[2],
    show=False,
    title='CD8A Expression',
    size=2,
    cmap='Blues',
    frameon=False
)

sc.pl.embedding(
    adata_tcell,
    basis='umap_scanvi_cd4cd8',
    color='CD8_score',
    ax=axes[3],
    show=False,
    title='CD8 Score (max CD8A/B)',
    size=2,
    cmap='Purples',
    frameon=False
)

plt.tight_layout()
plt.savefig(FIG_DIR / 'cd4_cd8_classification.pdf', dpi=300, bbox_inches='tight')
plt.close()
print(f"✓ Saved: cd4_cd8_classification.pdf")

### Plot 2: Combined Labels

In [ ]:
print(f"\nGenerating combined labels plot...")
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

sc.pl.embedding(
    adata,
    basis='umap_scanvi_cd4cd8',
    color='cell_type_combined_filt',
    ax=axes[0],
    show=False,
    title='Combined Labels (Filtered)',
    size=2,
    legend_loc='on data',
    legend_fontsize=5,
    frameon=False
)

sc.pl.embedding(
    adata,
    basis='umap_scanvi_cd4cd8',
    color='is_t_cell',
    ax=axes[1],
    show=False,
    title='T cell vs Non-T cell',
    size=2,
    frameon=False
)

sc.pl.embedding(
    adata,
    basis='umap_scanvi_cd4cd8',
    color=BATCH_KEY,
    ax=axes[2],
    show=False,
    title='Batch',
    size=1,
    frameon=False
)

plt.tight_layout()
plt.savefig(FIG_DIR / 'combined_labels.pdf', dpi=300, bbox_inches='tight')
plt.close()
print(f"✓ Saved: combined_labels.pdf")

### Plot 3: scANVI Final Results

In [ ]:
print(f"\nGenerating scANVI final results plot...")
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

sc.pl.embedding(
    adata,
    basis='umap_scanvi_cd4cd8',
    color='cell_type_scanvi_cd4cd8_filt',
    ax=axes[0],
    show=False,
    title='scANVI Final Annotations',
    size=2,
    legend_loc='on data',
    legend_fontsize=5,
    frameon=False
)

sc.pl.embedding(
    adata,
    basis='umap_scanvi_cd4cd8',
    color='scanvi_cd4cd8_confidence',
    ax=axes[1],
    show=False,
    title='scANVI Confidence',
    size=2,
    cmap='viridis',
    frameon=False
)

sc.pl.embedding(
    adata,
    basis='umap_scanvi_cd4cd8',
    color=CD4CD8_KEY,
    ax=axes[2],
    show=False,
    title='CD4/CD8 Type (winner-takes-all)',
    size=2,
    frameon=False
)

plt.tight_layout()
plt.savefig(FIG_DIR / 'scanvi_cd4cd8_final.pdf', dpi=300, bbox_inches='tight')
plt.close()
print(f"✓ Saved: scanvi_cd4cd8_final.pdf")

In [ ]:
# ============================================================================
# STAGE 10: MERGE scANVI LABELS (SIMPLIFY CELL TYPES)
# ============================================================================
# Merge similar cell types based on UMAP visualization
# Keep the most representative label for each group

print(f"\n{'='*80}")
print("STAGE 10: MERGE scANVI LABELS")
print("="*80)

# Define merge mapping
# Rule: First line in each group is the TARGET (keep), rest merge into it
merge_map = {
    # Group 1: CRTAM+ gamma-delta T cells
    'CD8+ CRTAM+ gamma-delta T cells': 'CRTAM+ gamma-delta T cells',
    
    # Group 2: Tem/Temra cytotoxic T cells (keep CD8+)
    'Tem/Temra cytotoxic T cells': 'CD8+ Tem/Temra cytotoxic T cells',
    
    # Group 3: gamma-delta T cells
    'CD8+ gamma-delta T cells': 'gamma-delta T cells',
    
    'CD8+ Tem/Trm cytotoxic T cells':'Tem/Trm cytotoxic T cells',
    
    # Group 4: Type 17 helper T cells
    'CD8+ Type 17 helper T cells': 'Type 17 helper T cells',
    
    # Group 5: Trm cytotoxic T cells (keep CD8+)
    'Trm cytotoxic T cells': 'CD8+ Trm cytotoxic T cells',
    
    # Group 6: Tem/Effector helper T cells (keep CD4+)
    'Tem/Effector helper T cells': 'CD4+ Tem/Effector helper T cells',
    
    # Group 7: MAIT cells
    'CD8+ MAIT cells': 'MAIT cells',
    
    # Group 8: NK cells (merge CD16+ and CD16-)
    'CD16+ NK cells': 'NK cells',
    'CD16- NK cells': 'NK cells',
    
    # Group 9: Regulatory T cells (merge CD4+ and CD8+)
    'CD8+ Regulatory T cells': 'Regulatory T cells',
    'CD4+ Regulatory T cells': 'Regulatory T cells',
}

print(f"\nMerge mapping defined:")
print(f"  Groups to merge: {len(set(merge_map.values()))}")
print(f"  Source labels: {len(merge_map)}")

# Apply mapping to scANVI labels
print(f"\nApplying merge mapping...")

source_col = 'cell_type_scanvi_cd4cd8_filt'
target_col = 'cell_type_merged'

# Create merged labels
adata.obs[target_col] = adata.obs[source_col].astype(str).replace(merge_map)

# Statistics
n_before = adata.obs[source_col].nunique()
n_after = adata.obs[target_col].nunique()
n_merged = n_before - n_after

print(f"\n✓ Merging complete:")
print(f"  Before: {n_before} unique types")
print(f"  After:  {n_after} unique types")
print(f"  Reduced by: {n_merged} types")

# Show which types were actually merged
print(f"\n{'='*80}")
print("MERGE DETAILS:")
print("="*80)

found_merges = []
not_found = []

for old, new in merge_map.items():
    n_old = (adata.obs[source_col] == old).sum()
    if n_old > 0:
        found_merges.append((old, new, n_old))
        print(f"\n{old}")
        print(f"  → {new}")
        print(f"  Cells: {n_old:,}")
    else:
        not_found.append(old)

if not_found:
    print(f"\n⚠️  Labels not found in data ({len(not_found)}):")
    for label in not_found:
        print(f"  - {label}")

# Count final merged types
print(f"\n{'='*80}")
print("FINAL MERGED CELL TYPE COUNTS:")
print("="*80)

merged_counts = adata.obs[target_col].value_counts()
for ct, count in merged_counts.items():
    pct = count / adata.n_obs * 100
    print(f"{ct}: {count:,} cells ({pct:.1f}%)")

# Save merge mapping for reference
merge_df = pd.DataFrame([
    {'original': k, 'merged': v} for k, v in merge_map.items()
])
merge_file = OUTPUT_DIR / "cell_type_merge_mapping.csv"
merge_df.to_csv(merge_file, index=False)
print(f"\n✓ Merge mapping saved: {merge_file.name}")

# Save merged counts
merged_counts_df = pd.DataFrame({
    'cell_type': merged_counts.index,
    'count': merged_counts.values,
    'percentage': (merged_counts.values / merged_counts.sum() * 100).round(2)
})
merged_counts_file = OUTPUT_DIR / "cell_type_merged_counts.csv"
merged_counts_df.to_csv(merged_counts_file, index=False)
print(f"✓ Merged counts saved: {merged_counts_file.name}")

print(f"\n{'='*80}")
print("✅ STAGE 10 COMPLETE")
print("="*80)

In [ ]:
# ============================================================================
# STAGE 11: VISUALIZE MERGED LABELS
# ============================================================================

print(f"\n{'='*80}")
print("STAGE 11: VISUALIZE MERGED LABELS")
print("="*80)

# ============================================================================
# Plot 1: Before vs After Merge (Side by Side)
# ============================================================================

print(f"\nGenerating before/after comparison...")

fig, axes = plt.subplots(1, 2, figsize=(24, 10))

# Before merge
sc.pl.embedding(
    adata,
    basis='umap_scanvi_cd4cd8',
    color='cell_type_scanvi_cd4cd8_filt',
    ax=axes[0],
    show=False,
    title=f'Before Merge ({adata.obs["cell_type_scanvi_cd4cd8_filt"].nunique()} types)',
    size=3,
    legend_loc='on data',
    legend_fontsize=6,
    frameon=False
)

# After merge
sc.pl.embedding(
    adata,
    basis='umap_scanvi_cd4cd8',
    color='cell_type_merged',
    ax=axes[1],
    show=False,
    title=f'After Merge ({adata.obs["cell_type_merged"].nunique()} types)',
    size=3,
    legend_loc='on data',
    legend_fontsize=7,
    frameon=False
)

plt.tight_layout()
plt.savefig(FIG_DIR / 'merge_before_after_comparison.pdf', dpi=300, bbox_inches='tight')
plt.close()
print(f"✓ Saved: merge_before_after_comparison.pdf")

# ============================================================================
# Plot 2: Merged Labels with Confidence
# ============================================================================

print(f"\nGenerating merged labels with confidence...")

fig, axes = plt.subplots(1, 3, figsize=(30, 9))

# Merged cell types
sc.pl.embedding(
    adata,
    basis='umap_scanvi_cd4cd8',
    color='cell_type_merged',
    ax=axes[0],
    show=False,
    title='Merged Cell Types',
    size=3,
    legend_loc='on data',
    legend_fontsize=7,
    frameon=False
)

# Confidence scores
sc.pl.embedding(
    adata,
    basis='umap_scanvi_cd4cd8',
    color='scanvi_cd4cd8_confidence',
    ax=axes[1],
    show=False,
    title='scANVI Confidence',
    size=2,
    cmap='viridis',
    frameon=False
)

# CD4/CD8 classification
sc.pl.embedding(
    adata,
    basis='umap_scanvi_cd4cd8',
    color='cd4_cd8_type',
    ax=axes[2],
    show=False,
    title='CD4/CD8 Classification',
    size=2,
    frameon=False
)

plt.tight_layout()
plt.savefig(FIG_DIR / 'merged_labels_with_confidence.pdf', dpi=300, bbox_inches='tight')
plt.close()
print(f"✓ Saved: merged_labels_with_confidence.pdf")

# ============================================================================
# Plot 3: Specific Merged Groups (Show merge effect)
# ============================================================================

print(f"\nGenerating specific merge examples...")

# Find which types were actually merged
merged_groups = {}
for old, new in merge_map.items():
    if (adata.obs['cell_type_scanvi_cd4cd8_filt'] == old).sum() > 0:
        if new not in merged_groups:
            merged_groups[new] = []
        merged_groups[new].append(old)

# Filter to groups with actual merges (>1 source)
significant_merges = {k: v for k, v in merged_groups.items() if len(v) > 1}

if len(significant_merges) > 0:
    print(f"\nShowing {len(significant_merges)} groups with actual merges:")
    
    n_rows = (len(significant_merges) + 1) // 2
    fig, axes = plt.subplots(n_rows, 2, figsize=(20, 6*n_rows))
    axes = axes.flatten() if n_rows > 1 else [axes] if len(significant_merges) == 1 else axes
    
    for idx, (target, sources) in enumerate(significant_merges.items()):
        if idx >= len(axes):
            break
        
        # Highlight cells in this merged group
        is_in_group = adata.obs['cell_type_merged'] == target
        highlight = pd.Series(['Other'] * adata.n_obs, index=adata.obs_names)
        highlight[is_in_group] = target
        
        n_cells = is_in_group.sum()
        
        # ⭐ FIX: Add highlight to adata.obs temporarily to avoid ax conflict
        temp_col = f'_temp_highlight_{idx}'
        adata.obs[temp_col] = highlight
        
        sc.pl.embedding(
            adata,
            basis='umap_scanvi_cd4cd8',
            color=temp_col,  # Use column name instead of Series
            ax=axes[idx],
            show=False,
            title=f'{target}\n({n_cells:,} cells from {len(sources)} labels)',
            size=3,
            palette=['lightgray', 'red'],
            frameon=False
        )
        
        # Clean up temporary column
        adata.obs.drop(columns=[temp_col], inplace=True)
        
        # Add text annotation
        sources_str = '\n'.join([f'  • {s}' for s in sources])
        print(f"\n{idx+1}. {target} ({n_cells:,} cells)")
        print(f"   Merged from:")
        for s in sources:
            n = (adata.obs['cell_type_scanvi_cd4cd8_filt'] == s).sum()
            print(f"     • {s}: {n:,} cells")
    
    # Hide unused axes
    for idx in range(len(significant_merges), len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'merge_group_details.pdf', dpi=300, bbox_inches='tight')
    plt.close()
    print(f"\n✓ Saved: merge_group_details.pdf")
else:
    print(f"\n⚠️  No multi-source merges found")

# ============================================================================
# Plot 4: Cell Type Proportions Bar Chart
# ============================================================================

print(f"\nGenerating proportion comparison...")

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Before merge
before_counts = adata.obs['cell_type_scanvi_cd4cd8_filt'].value_counts()
before_pct = (before_counts / adata.n_obs * 100).sort_values(ascending=True)

axes[0].barh(range(len(before_pct)), before_pct.values)
axes[0].set_yticks(range(len(before_pct)))
axes[0].set_yticklabels(before_pct.index, fontsize=8)
axes[0].set_xlabel('Percentage of cells (%)')
axes[0].set_title(f'Before Merge ({len(before_pct)} types)')
axes[0].grid(axis='x', alpha=0.3)

# After merge
after_counts = adata.obs['cell_type_merged'].value_counts()
after_pct = (after_counts / adata.n_obs * 100).sort_values(ascending=True)

axes[1].barh(range(len(after_pct)), after_pct.values, color='coral')
axes[1].set_yticks(range(len(after_pct)))
axes[1].set_yticklabels(after_pct.index, fontsize=9)
axes[1].set_xlabel('Percentage of cells (%)')
axes[1].set_title(f'After Merge ({len(after_pct)} types)')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / 'merge_proportions_comparison.pdf', dpi=300, bbox_inches='tight')
plt.close()
print(f"✓ Saved: merge_proportions_comparison.pdf")

# ============================================================================
# Summary Statistics
# ============================================================================

print(f"\n{'='*80}")
print("MERGE VISUALIZATION SUMMARY")
print("="*80)

print(f"\n📊 Cell Type Changes:")
print(f"  Before merge: {len(before_counts)} types")
print(f"  After merge:  {len(after_counts)} types")
print(f"  Reduced by:   {len(before_counts) - len(after_counts)} types ({(len(before_counts) - len(after_counts))/len(before_counts)*100:.1f}%)")

print(f"\n📈 Top 5 cell types (after merge):")
for ct, count in after_counts.head(5).items():
    pct = count / adata.n_obs * 100
    print(f"  {ct}: {count:,} cells ({pct:.1f}%)")

print(f"\n📁 Figures saved:")
print(f"  - merge_before_after_comparison.pdf (side-by-side UMAP)")
print(f"  - merged_labels_with_confidence.pdf (merged + confidence)")
print(f"  - merge_group_details.pdf (individual merge groups)")
print(f"  - merge_proportions_comparison.pdf (bar charts)")

print(f"\n{'='*80}")
print("✅ STAGE 11 COMPLETE")
print("="*80)

In [ ]:
# ============================================================================
# STAGE 13: FILTER ENDOTHELIAL CELLS & RETRAIN scANVI
# ============================================================================

print(f"\n{'='*80}")
print("STAGE 13: FILTER ENDOTHELIAL & RETRAIN scANVI")
print("="*80)

# ============================================================================
# Step 1: Filter out Endothelial cells
# ============================================================================

print(f"\nStep 1: Filtering Endothelial cells...")

# Count before
n_before = adata.n_obs
endo_mask = adata.obs['cell_type_merged'].str.contains('Endothelial', case=False, na=False)
n_endo = endo_mask.sum()

print(f"  Total cells before: {n_before:,}")
print(f"  Endothelial cells: {n_endo:,} ({n_endo/n_before*100:.1f}%)")

# Filter
adata_filtered = adata[~endo_mask].copy()
n_after = adata_filtered.n_obs

print(f"  Cells after filtering: {n_after:,}")
print(f"  Removed: {n_before - n_after:,} cells")

# Check remaining cell types
print(f"\n✓ Remaining cell types ({adata_filtered.obs['cell_type_merged'].nunique()}):")
remaining_types = adata_filtered.obs['cell_type_merged'].value_counts()
for ct, count in remaining_types.items():
    pct = count / n_after * 100
    print(f"  {ct}: {count:,} cells ({pct:.1f}%)")

# ============================================================================
# Step 2: Prepare for scANVI retraining
# ============================================================================

print(f"\n{'='*80}")
print(f"Step 2: Preparing data for scANVI retraining...")
print("="*80)

# Filter rare types in merged labels
print(f"\nFiltering rare types (min {MIN_CELLS_PER_TYPE} cells)...")
cell_type_counts = adata_filtered.obs['cell_type_merged'].value_counts()
rare_types = cell_type_counts[cell_type_counts < MIN_CELLS_PER_TYPE].index.tolist()

if len(rare_types) > 0:
    print(f"  Rare types found: {len(rare_types)}")
    for rt in rare_types:
        print(f"    - {rt}: {cell_type_counts[rt]} cells")
    
    # Create filtered label column
    adata_filtered.obs['cell_type_merged_filt'] = adata_filtered.obs['cell_type_merged'].copy()
    adata_filtered.obs.loc[
        adata_filtered.obs['cell_type_merged_filt'].isin(rare_types),
        'cell_type_merged_filt'
    ] = 'Unknown'
    
    print(f"✓ Rare types merged to 'Unknown'")
else:
    print(f"✓ No rare types found")
    adata_filtered.obs['cell_type_merged_filt'] = adata_filtered.obs['cell_type_merged'].copy()

# Final label counts
final_labels = adata_filtered.obs['cell_type_merged_filt'].value_counts()
n_final_types = len(final_labels)
print(f"\n✓ Final cell types for scANVI: {n_final_types}")
for ct, count in final_labels.items():
    pct = count / adata_filtered.n_obs * 100
    print(f"  {ct}: {count:,} cells ({pct:.1f}%)")

# ============================================================================
# Step 3: Prepare adata_model for scANVI
# ============================================================================

print(f"\n{'='*80}")
print(f"Step 3: Creating adata_model for training...")
print("="*80)

# Check if we have HVG mask
if 'highly_variable' not in adata_filtered.var.columns:
    print(f"⚠️  'highly_variable' column missing, using all genes from original adata")
    # Use the same genes as original training
    hvg_genes_orig = adata.var_names[adata.var['highly_variable']].tolist()
    # Subset filtered adata
    adata_model_filt = adata_filtered[:, hvg_genes_orig].copy()
else:
    hvg_mask_filt = adata_filtered.var['highly_variable'].values
    adata_model_filt = adata_filtered[:, hvg_mask_filt].copy()

print(f"✓ adata_model created:")
print(f"  Cells: {adata_model_filt.n_obs:,}")
print(f"  Genes: {adata_model_filt.n_vars:,}")

# Ensure counts layer exists
if 'counts' not in adata_model_filt.layers:
    print(f"⚠️  'counts' layer missing, using .X")
    adata_model_filt.layers['counts'] = adata_model_filt.X.copy()

# Setup scANVI
print(f"\nSetting up scANVI...")
scvi.model.SCANVI.setup_anndata(
    adata_model_filt,
    batch_key=BATCH_KEY,
    labels_key='cell_type_merged_filt',
    layer='counts',
    unlabeled_category='Unknown'
)
print(f"✓ scANVI setup complete")

# ============================================================================
# Step 4: Train scANVI
# ============================================================================

print(f"\n{'='*80}")
print(f"Step 4: Training scANVI on filtered data...")
print("="*80)

# Check if we can load pretrained scVI
scvi_base_model = None
if SCVI_MODEL_PATH is not None and Path(SCVI_MODEL_PATH).exists():
    print(f"\nAttempting to load pre-trained scVI model...")
    try:
        scvi_base_model = scvi.model.SCVI.load(SCVI_MODEL_PATH, adata=adata_model_filt)
        print(f"✓ Loaded pre-trained scVI model")
    except Exception as e:
        print(f"⚠️  Failed to load scVI model: {str(e)[:100]}")
        print(f"   Will train scANVI from scratch")

# Initialize scANVI
if scvi_base_model is not None:
    print(f"\nInitializing scANVI from pre-trained scVI...")
    scanvi_model = scvi.model.SCANVI.from_scvi_model(
        scvi_base_model,
        unlabeled_category='Unknown',
        labels_key='cell_type_merged_filt'
    )
else:
    print(f"\nInitializing scANVI from scratch...")
    scanvi_model = scvi.model.SCANVI(
        adata_model_filt,
        unlabeled_category='Unknown',
        n_latent=N_LATENT,
        n_layers=N_LAYERS
    )

# Train
print(f"\nTraining scANVI...")
print(f"  Max epochs: {SCANVI_MAX_EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Early stopping: {EARLY_STOPPING}")

scanvi_model.train(
    max_epochs=SCANVI_MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    early_stopping=EARLY_STOPPING,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    train_size=0.9
)

print(f"✓ Training complete")

# ============================================================================
# Step 5: Extract scANVI results
# ============================================================================

print(f"\n{'='*80}")
print(f"Step 5: Extracting scANVI results...")
print("="*80)

# Get predictions
print(f"\nExtracting predictions...")
predictions = scanvi_model.predict()
soft_predictions = scanvi_model.predict(soft=True)
confidence = np.asarray(soft_predictions.max(axis=1)).flatten()

# Get latent representation
print(f"Extracting latent representation...")
latent = scanvi_model.get_latent_representation()

# Store in filtered adata
adata_filtered.obs['cell_type_scanvi_final'] = predictions
adata_filtered.obs['scanvi_final_confidence'] = confidence
adata_filtered.obsm['X_scanvi_final'] = latent

print(f"✓ Results extracted:")
print(f"  Predictions: {len(predictions):,}")
print(f"  Unique types: {len(set(predictions))}")
print(f"  Mean confidence: {confidence.mean():.3f}")
print(f"  Median confidence: {np.median(confidence):.3f}")

# Compute UMAP on scANVI latent
print(f"\nComputing UMAP on scANVI latent space...")
sc.pp.neighbors(adata_filtered, use_rep='X_scanvi_final', key_added='scanvi_final')
sc.tl.umap(adata_filtered, neighbors_key='scanvi_final')
adata_filtered.obsm['umap_scanvi_final'] = adata_filtered.obsm['X_umap'].copy()
print(f"✓ UMAP computed")

# Save model
scanvi_model_dir = OUTPUT_DIR / "models" / "scanvi_final_model"
scanvi_model_dir.mkdir(parents=True, exist_ok=True)
scanvi_model.save(scanvi_model_dir, overwrite=True)
print(f"\n✓ Model saved: {scanvi_model_dir}")

print(f"\n{'='*80}")
print("✅ STAGE 13 COMPLETE")
print("="*80)

In [ ]:
# ============================================================================
# STAGE 14: CALCULATE MARKER GENES
# ============================================================================

print(f"\n{'='*80}")
print("STAGE 14: CALCULATE MARKER GENES")
print("="*80)

# ============================================================================
# Step 1: Prepare for marker gene calculation
# ============================================================================

print(f"\nPreparing for marker gene calculation...")

# Check if we should use raw or current X
use_raw = adata_filtered.raw is not None

if use_raw:
    print(f"✓ Will use adata.raw ({adata_filtered.raw.n_vars:,} genes)")
else:
    print(f"✓ Will use current adata.X ({adata_filtered.n_vars:,} genes)")

# Check cell type column
marker_groupby = 'cell_type_scanvi_final'
cell_types = adata_filtered.obs[marker_groupby].unique()
n_types = len(cell_types)

print(f"\nCell types for marker calculation: {n_types}")
for ct in sorted(cell_types):
    n = (adata_filtered.obs[marker_groupby] == ct).sum()
    print(f"  {ct}: {n:,} cells")

# ============================================================================
# Step 2: Calculate marker genes (Wilcoxon)
# ============================================================================

print(f"\n{'='*80}")
print(f"Calculating marker genes (Wilcoxon rank-sum test)...")
print("="*80)

sc.tl.rank_genes_groups(
    adata_filtered,
    groupby=marker_groupby,
    method='wilcoxon',
    use_raw=use_raw,
    key_added='rank_genes_scanvi_final'
)

print(f"✓ Marker genes calculated")

# ============================================================================
# Step 3: Extract and save marker genes
# ============================================================================

print(f"\n{'='*80}")
print(f"Extracting top markers for each cell type...")
print("="*80)

# Extract top markers
n_top_genes = 100  # Top 100 markers per cell type

marker_results = []

for cell_type in cell_types:
    # Get markers for this cell type
    markers = sc.get.rank_genes_groups_df(
        adata_filtered,
        group=cell_type,
        key='rank_genes_scanvi_final'
    )
    
    # Filter significant markers
    markers_sig = markers[markers['pvals_adj'] < 0.05].copy()
    
    # Add cell type
    markers_sig['cell_type'] = cell_type
    
    # Keep top N
    markers_top = markers_sig.head(n_top_genes)
    
    marker_results.append(markers_top)
    
    print(f"\n{cell_type}:")
    print(f"  Total significant: {len(markers_sig):,}")
    print(f"  Top 5 markers:")
    for i, row in markers_top.head(5).iterrows():
        print(f"    {row['names']}: logFC={row['logfoldchanges']:.2f}, p_adj={row['pvals_adj']:.2e}")

# Combine all markers
all_markers = pd.concat(marker_results, ignore_index=True)

print(f"\n✓ Total marker genes: {len(all_markers):,}")

# Save markers
markers_file = OUTPUT_DIR / "markers_scanvi_final_top100.csv"
all_markers.to_csv(markers_file, index=False)
print(f"✓ Saved: {markers_file.name}")

# ============================================================================
# Step 4: Create marker heatmap
# ============================================================================

print(f"\n{'='*80}")
print(f"Generating marker gene heatmap...")
print("="*80)

# Select top 5 markers per cell type for heatmap
top_markers = {}
for cell_type in cell_types:
    ct_markers = all_markers[all_markers['cell_type'] == cell_type]
    top5 = ct_markers.head(5)['names'].tolist()
    top_markers[cell_type] = top5

# Flatten to list
marker_genes = []
for ct, genes in top_markers.items():
    marker_genes.extend(genes)

# Remove duplicates while preserving order
marker_genes_unique = []
seen = set()
for gene in marker_genes:
    if gene not in seen:
        marker_genes_unique.append(gene)
        seen.add(gene)

print(f"  Unique marker genes for heatmap: {len(marker_genes_unique)}")

# Check if markers exist in data
if use_raw:
    available_markers = [g for g in marker_genes_unique if g in adata_filtered.raw.var_names]
else:
    available_markers = [g for g in marker_genes_unique if g in adata_filtered.var_names]

print(f"  Available in data: {len(available_markers)}")

if len(available_markers) > 0:
    # Create heatmap
    fig_file = FIG_DIR / 'marker_heatmap_scanvi_final.pdf'
    
    sc.pl.heatmap(
        adata_filtered,
        var_names=available_markers,
        groupby=marker_groupby,
        use_raw=use_raw,
        swap_axes=True,
        show_gene_labels=True,
        figsize=(12, 8),
        dendrogram=False,
        save=False
    )
    
    plt.savefig(fig_file, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved: {fig_file.name}")
else:
    print(f"⚠️  No markers available for heatmap")

# ============================================================================
# Step 5: Create marker dotplot
# ============================================================================

print(f"\nGenerating marker dotplot...")

if len(available_markers) > 0 and len(available_markers) <= 50:
    fig_file = FIG_DIR / 'marker_dotplot_scanvi_final.pdf'
    
    sc.pl.dotplot(
        adata_filtered,
        var_names=available_markers,
        groupby=marker_groupby,
        use_raw=use_raw,
        dendrogram=False,
        figsize=(14, len(cell_types) * 0.4),
        save=False
    )
    
    plt.savefig(fig_file, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved: {fig_file.name}")
elif len(available_markers) > 50:
    print(f"⚠️  Too many markers ({len(available_markers)}) for dotplot, skipping")
else:
    print(f"⚠️  No markers for dotplot")

# ============================================================================
# Step 6: Visualize final scANVI results
# ============================================================================

print(f"\n{'='*80}")
print(f"Visualizing final scANVI results...")
print("="*80)

fig, axes = plt.subplots(1, 3, figsize=(30, 9))

# Final cell types
sc.pl.embedding(
    adata_filtered,
    basis='umap_scanvi_final',
    color=marker_groupby,
    ax=axes[0],
    show=False,
    title=f'Final Cell Types (No Endothelial)\n{n_types} types',
    size=3,
    legend_loc='on data',
    legend_fontsize=7,
    frameon=False
)

# Confidence
sc.pl.embedding(
    adata_filtered,
    basis='umap_scanvi_final',
    color='scanvi_final_confidence',
    ax=axes[1],
    show=False,
    title='scANVI Confidence',
    size=2,
    cmap='viridis',
    frameon=False
)

# CD4/CD8 (if available)
if 'cd4_cd8_type' in adata_filtered.obs.columns:
    sc.pl.embedding(
        adata_filtered,
        basis='umap_scanvi_final',
        color='cd4_cd8_type',
        ax=axes[2],
        show=False,
        title='CD4/CD8 Classification',
        size=2,
        frameon=False
    )
else:
    # Show batch
    sc.pl.embedding(
        adata_filtered,
        basis='umap_scanvi_final',
        color=BATCH_KEY,
        ax=axes[2],
        show=False,
        title='Batch',
        size=2,
        frameon=False
    )

plt.tight_layout()
fig_file = FIG_DIR / 'scanvi_final_umap.pdf'
plt.savefig(fig_file, dpi=300, bbox_inches='tight')
plt.close()
print(f"✓ Saved: {fig_file.name}")

print(f"\n{'='*80}")
print("✅ STAGE 14 COMPLETE")
print("="*80)

In [ ]:
# ============================================================================
# STAGE 15: SAVE FINAL RESULTS (NO ENDOTHELIAL)
# ============================================================================

print(f"\n{'='*80}")
print("STAGE 15: SAVE FINAL RESULTS")
print("="*80)

output_file = OUTPUT_DIR / "adata_tcell_FINAL_no_endo.h5ad"

# Update metadata
adata_filtered.uns['pipeline_info'] = {
    'version': '1.3_PRODUCTION_MERGED_NO_ENDO',
    'date': '2025-01-09',
    'input_file': str(INPUT_FILE),
    'batch_key': BATCH_KEY,
    'cd4_cd8_threshold': CD4CD8_THRESHOLD,
    'cd4_cd8_strategy': 'winner-takes-all',
    'min_cells_per_type': MIN_CELLS_PER_TYPE,
    'exclude_keywords': EXCLUDE_KEYWORDS,
    't_cell_keywords': T_CELL_KEYWORDS,
    'filtering': {
        'removed_endothelial': True,
        'cells_before': n_before,
        'cells_after': n_after,
        'removed': n_before - n_after
    },
    'label_merging': {
        'applied': True,
        'n_groups': len(set(merge_map.values())),
        'n_merged': len(merge_map),
        'merge_map': merge_map
    },
    'scanvi_retrain': {
        'applied': True,
        'n_cell_types': n_types,
        'model_path': str(scanvi_model_dir)
    },
    'marker_genes': {
        'method': 'wilcoxon',
        'n_top_per_type': n_top_genes,
        'use_raw': use_raw
    },
    'critical_features': [
        'Endothelial cells removed',
        'scANVI retrained on merged labels',
        'Marker genes calculated (top 100 per type)',
        'Winner-takes-all CD4/CD8 classification',
        'Label merging applied',
        'Full gene set preserved in .raw'
    ]
}

# Save
print(f"\nSaving final h5ad file...")
adata_filtered.write_h5ad(output_file, compression='gzip')
size_gb = output_file.stat().st_size / 1e9
print(f"✓ Saved: {output_file.name}")
print(f"   Size: {size_gb:.2f} GB")

# Export complete annotations
print(f"\nExporting complete annotations...")
annotation_cols = [
    CELLTYPIST_KEY,
    'is_t_cell',
    CD4CD8_KEY,
    'cell_type_merged',
    'cell_type_merged_filt',
    'cell_type_scanvi_final',
    'scanvi_final_confidence',
    BATCH_KEY
]

# Filter to existing columns
annotation_cols = [col for col in annotation_cols if col in adata_filtered.obs.columns]

annotations = adata_filtered.obs[annotation_cols].copy()
annotations_file = OUTPUT_DIR / "annotations_final_no_endo.csv"
annotations.to_csv(annotations_file)
print(f"✓ Annotations exported: {annotations_file.name}")

# Export cell type counts
print(f"\nExporting cell type counts...")

# Final scANVI counts
scanvi_counts = adata_filtered.obs['cell_type_scanvi_final'].value_counts()
scanvi_counts_df = pd.DataFrame({
    'cell_type': scanvi_counts.index,
    'count': scanvi_counts.values,
    'percentage': (scanvi_counts.values / scanvi_counts.sum() * 100).round(2)
})
scanvi_counts_file = OUTPUT_DIR / "cell_type_scanvi_final_counts.csv"
scanvi_counts_df.to_csv(scanvi_counts_file, index=False)
print(f"✓ Saved: {scanvi_counts_file.name}")

# Merged counts
merged_counts = adata_filtered.obs['cell_type_merged'].value_counts()
merged_counts_df = pd.DataFrame({
    'cell_type': merged_counts.index,
    'count': merged_counts.values,
    'percentage': (merged_counts.values / merged_counts.sum() * 100).round(2)
})
merged_counts_file = OUTPUT_DIR / "cell_type_merged_counts_no_endo.csv"
merged_counts_df.to_csv(merged_counts_file, index=False)
print(f"✓ Saved: {merged_counts_file.name}")

# Export marker gene summary
print(f"\nCreating marker gene summary...")

# Top 10 markers per cell type
marker_summary = []
for ct in cell_types:
    ct_markers = all_markers[all_markers['cell_type'] == ct].head(10)
    marker_summary.append({
        'cell_type': ct,
        'n_cells': (adata_filtered.obs['cell_type_scanvi_final'] == ct).sum(),
        'top_markers': ', '.join(ct_markers['names'].tolist()),
        'n_significant': len(all_markers[all_markers['cell_type'] == ct])
    })

marker_summary_df = pd.DataFrame(marker_summary)
marker_summary_file = OUTPUT_DIR / "marker_summary_scanvi_final.csv"
marker_summary_df.to_csv(marker_summary_file, index=False)
print(f"✓ Saved: {marker_summary_file.name}")

print(f"\n{'='*80}")
print("✅ STAGE 15 COMPLETE")
print("="*80)

# ============================================================================
# FINAL SUMMARY
# ============================================================================

print(f"\n{'='*80}")
print("🎉 COMPLETE PIPELINE FINISHED")
print("="*80)

print(f"\n📊 Final Data Summary:")
print(f"   Cells: {adata_filtered.n_obs:,}")
print(f"   Genes: {adata_filtered.n_vars:,}")
print(f"   Batches: {adata_filtered.obs[BATCH_KEY].nunique()}")

print(f"\n⭐ Filtering:")
print(f"   Original cells: {n_before:,}")
print(f"   Endothelial removed: {n_endo:,} ({n_endo/n_before*100:.1f}%)")
print(f"   Final cells: {n_after:,}")

print(f"\n⭐ T cells vs Non-T cells:")
if 'is_t_cell' in adata_filtered.obs.columns:
    is_t_summary = adata_filtered.obs['is_t_cell'].value_counts()
    for val, count in is_t_summary.items():
        label = "T cells" if val else "Non-T cells (e.g., NK)"
        pct = count / adata_filtered.n_obs * 100
        print(f"   {label}: {count:,} cells ({pct:.1f}%)")

print(f"\n⭐ CD4/CD8 Classification:")
if CD4CD8_KEY in adata_filtered.obs.columns and 'is_t_cell' in adata_filtered.obs.columns:
    tcell_cd4cd8 = adata_filtered.obs.loc[adata_filtered.obs['is_t_cell'], CD4CD8_KEY].value_counts()
    n_tcells_total = adata_filtered.obs['is_t_cell'].sum()
    for cat in ['CD4+', 'CD8+', '']:
        if cat in tcell_cd4cd8.index:
            count = tcell_cd4cd8[cat]
            pct = count / n_tcells_total * 100
            label = cat if cat else 'Unclear'
            print(f"   {label}: {count:,} cells ({pct:.1f}% of T cells)")

print(f"\n⭐ Final Cell Types ({n_types}):")
for i, (ct, count) in enumerate(scanvi_counts.head(15).items(), 1):
    pct = count / adata_filtered.n_obs * 100
    print(f"   {i:2d}. {ct:40s} {count:6,} cells ({pct:5.1f}%)")

print(f"\n⭐ scANVI Performance:")
print(f"   Mean confidence: {adata_filtered.obs['scanvi_final_confidence'].mean():.3f}")
print(f"   Median confidence: {adata_filtered.obs['scanvi_final_confidence'].median():.3f}")

print(f"\n📁 Key Output Files:")
print(f"   Main data: adata_tcell_FINAL_no_endo.h5ad")
print(f"   Annotations: annotations_final_no_endo.csv")
print(f"   Markers (full): markers_scanvi_final_top100.csv")
print(f"   Markers (summary): marker_summary_scanvi_final.csv")
print(f"   scANVI model: models/scanvi_final_model/")

print(f"\n📈 Visualization Files:")
print(f"   - marker_heatmap_scanvi_final.pdf")
print(f"   - marker_dotplot_scanvi_final.pdf")
print(f"   - scanvi_final_umap.pdf")
print(f"   - merge_before_after_comparison.pdf")
print(f"   - merged_labels_with_confidence.pdf")

print(f"\n🔑 Recommended Columns:")
print(f"   - cell_type_merged: Simplified merged labels")
print(f"   - cell_type_scanvi_final: Final scANVI predictions ⭐")
print(f"   - scanvi_final_confidence: Prediction confidence")

print("\n" + "="*80)
print("✨ Analysis Complete - Ready for Publication!")
print("="*80)

print(f"\n💡 Next Steps:")
print(f"   1. Review marker_heatmap_scanvi_final.pdf")
print(f"   2. Check scanvi_final_umap.pdf for clustering quality")
print(f"   3. Use markers_scanvi_final_top100.csv for validation")
print(f"   4. Perform downstream analyses (DE, trajectory, etc.)")

print(f"\n✅ All outputs saved in: {OUTPUT_DIR}")

## STAGE 9: Save Final Results

In [ ]:
print(f"\n{'='*80}")
print("STAGE 9: SAVE FINAL RESULTS")
print("="*80)

output_file = OUTPUT_DIR / "adata_tcell_cd4cd8_FINAL.h5ad"

# Metadata
adata.uns['pipeline_info'] = {
    'version': '1.3_PRODUCTION',
    'date': '2025-01-09',
    'input_file': str(INPUT_FILE),
    'batch_key': BATCH_KEY,
    'cd4_cd8_threshold': CD4CD8_THRESHOLD,
    'cd4_cd8_strategy': 'winner-takes-all',
    'min_cells_per_type': MIN_CELLS_PER_TYPE,
    'exclude_keywords': EXCLUDE_KEYWORDS,
    't_cell_keywords': T_CELL_KEYWORDS,
    'cd8_score': 'max(CD8A, CD8B)',
    'scvi_model_source': str(SCVI_MODEL_PATH),
    'critical_fixes_v1_3': [
        'Winner-takes-all CD4/CD8 (no DP/DN prefixes)',
        'log1p layer created in Stage 0',
        'HVG consistency with pre-trained model',
        'Regex-escaped keyword matching',
        'Robust gene matching (handle duplicates)'
    ]
}

# Save
print(f"\nSaving final h5ad file...")
adata.write_h5ad(output_file, compression='gzip')
size_gb = output_file.stat().st_size / 1e9
print(f"✓ Saved: {output_file}")
print(f"   Size: {size_gb:.2f} GB")

# Export annotations
print(f"\nExporting annotations...")
annotations_cols = [
    CELLTYPIST_KEY,
    'is_t_cell',
    CD4CD8_KEY,
    'CD4_expr',
    'CD8A_expr',
    'CD8_score',
    COMBINED_LABEL_KEY,
    'cell_type_combined_filt',
    SCANVI_LABEL_KEY,
    'cell_type_scanvi_cd4cd8_filt',
    'scanvi_cd4cd8_confidence',
    BATCH_KEY
]

if 'CD8B_expr' in adata.obs.columns:
    annotations_cols.insert(5, 'CD8B_expr')

annotations = adata.obs[annotations_cols].copy()
annotations.to_csv(OUTPUT_DIR / "annotations_cd4cd8_complete.csv")
print(f"✓ Annotations exported")

## Final Summary

In [ ]:
print(f"\n{'='*80}")
print("🎉 PIPELINE COMPLETE - v1.3 PRODUCTION")
print("="*80)

print(f"\n📊 Analysis Summary:")
print(f"   Cells (final): {adata.n_obs:,}")
print(f"   Genes: {adata.n_vars:,}")
print(f"   Batches: {adata.obs[BATCH_KEY].nunique()}")

print(f"\n⭐ T cells vs Non-T cells:")
is_t_summary = adata.obs['is_t_cell'].value_counts()
for val, count in is_t_summary.items():
    label = "T cells" if val else "Non-T cells (e.g., NK)"
    pct = count / adata.n_obs * 100
    print(f"   {label}: {count:,} cells ({pct:.1f}%)")

print(f"\n⭐ CD4/CD8 Classification (T cells, winner-takes-all):")
tcell_cd4cd8 = adata.obs.loc[adata.obs['is_t_cell'], CD4CD8_KEY].value_counts()
n_tcells_total = adata.obs['is_t_cell'].sum()
for cat in ['CD4+', 'CD8+', '']:
    if cat in tcell_cd4cd8.index:
        count = tcell_cd4cd8[cat]
        pct = count / n_tcells_total * 100
        label = cat if cat else 'Unclear (DP/DN/low)'
        print(f"   {label}: {count:,} cells ({pct:.1f}% of T cells)")

print(f"\n⭐ Combined Labels:")
print(f"   Unique types (raw): {adata.obs[COMBINED_LABEL_KEY].nunique()}")
print(f"   Unique types (filtered): {adata.obs['cell_type_combined_filt'].nunique()}")

print(f"\n⭐ scANVI Results:")
print(f"   Unique types (raw): {adata.obs[SCANVI_LABEL_KEY].nunique()}")
print(f"   Unique types (filtered): {adata.obs['cell_type_scanvi_cd4cd8_filt'].nunique()}")
print(f"   Mean confidence: {float(np.mean(adata.obs['scanvi_cd4cd8_confidence'])):.3f}")

print(f"\n📁 Output Files:")
print(f"   h5ad: {output_file.name}")
print(f"   Annotations: annotations_cd4cd8_complete.csv")
print(f"   Counts: combined_labels_*.csv, scanvi_cd4cd8_counts_*.csv")
print(f"   Model: scanvi_cd4cd8_model/")

print(f"\n📈 Figures:")
print(f"   cd4_cd8_classification.pdf")
print(f"   combined_labels.pdf")
print(f"   scanvi_cd4cd8_final.pdf")

print("="*80)
print("\n✨ Production Fixes in v1.3:")
print("   ✅ Winner-takes-all CD4/CD8 (no label explosion)")
print("   ✅ log1p layer created early (consistent thresholds)")
print("   ✅ HVG consistency check with pre-trained model")
print("   ✅ Safe regex keyword matching")
print("   ✅ Robust gene matching (handles duplicates)")
print("   ✅ Memory-efficient plotting (views not copies)")
print("   ✅ Headless server compatible (Agg backend)")